# **Models from scratch**

[<font color='steelblue'>1. - __EDA__</font>](#one-bullet) <br>
    [<font color='steelblue'>1.1. - Import Libraries</font>](#two-bullet) <br>
    [<font color='steelblue'>1.2. - Import Data Files</font>](#three-bullet) <br>
    [<font color='steelblue'>1.3. - PreProcessing</font>](#four-bullet) <br>
    [<font color='steelblue'>1.4. - Augmentation</font>](#five-bullet) <br>
    [<font color='steelblue'>1.5. - Class weights</font>](#six-bullet) <br>

[<font color='steelblue'>2. - __Models__</font>](#seven-bullet) <br>
    [<font color='steelblue'>2.1. - PreModel</font>](#eight-bullet) <br>
    [<font color='steelblue'>2.2. - Sequential</font>](#nine-bullet) <br>
    [<font color='steelblue'>2.3. - Functional</font>](#ten-bullet) <br>
    [<font color='steelblue'>2.4. - Functional with more layers</font>](#eleven-bullet) <br>

Group 07
|Name | Number |
|----|----|
|Fábio dos Santos| 20240678|
|Joana Rodrigues| 20240603|
|Mara Simões| 20240326|
|Matilde Street| 20240523|
|Rafael Borges| 20240497|

<hr>
<a class="anchor" id="one-bullet"> 
<d style="color:white;">

## 1. EDA
</a> 
</d>   

<a class="anchor" id="two-bullet"> 
<d style="color:white;">

### 1.1. Import Libraries
</a> 
</d>   

In [177]:
import math
from pathlib import Path
import numpy as np
import pandas as pd
import pickle

# Visualization
import matplotlib.pyplot as plt
import plotly.express as px

# TensorFlow and Keras
import tensorflow as tf
from tensorflow import add
import tensorflow.keras as keras
from tensorflow.keras import layers, models, optimizers, losses
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import CategoricalAccuracy, AUC, SparseCategoricalAccuracy
from keras.layers import LeakyReLU
from keras.utils import to_categorical
from keras.metrics import F1Score
from keras.optimizers import Adam, RMSprop
import keras.backend as K
from tensorflow.keras.callbacks import EarlyStopping

# Preprocessing 
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import class_weight

# Augmentation
from keras.layers import RandomBrightness, RandomContrast, RandomFlip, RandomRotation, RandomSaturation, Pipeline

# Bayesian optimization
#from keras_tuner import BayesianOptimization
#from keras.callbacks import Callback, EarlyStopping
#import wandb
#from wandb.integration.keras import WandbCallback

<a class="anchor" id="three-bullet"> 
<d style="color:white;">

### 1.2. Import Data Files
</a> 
</d>  

Firstly we create the path and after we will load the data files

In [ ]:
path = Path('.') / 'data' / 'metadata.csv'

df = pd.read_csv(path)
df

,rare_species_id,eol_content_id,eol_page_id,kingdom,phylum,family,file_path
0,75fd91cb-2881-41cd-88e6-de451e8b60e2,12853737,449393,animalia,mollusca,unionidae,mollusca_unionidae/12853737_449393_eol-full-si...
1,28c508bc-63ff-4e60-9c8f-1934367e1528,20969394,793083,animalia,chordata,geoemydidae,chordata_geoemydidae/20969394_793083_eol-full-...
2,00372441-588c-4af8-9665-29bee20822c0,28895411,319982,animalia,chordata,cryptobranchidae,chordata_cryptobranchidae/28895411_319982_eol-...
3,29cc6040-6af2-49ee-86ec-ab7d89793828,29658536,45510188,animalia,chordata,turdidae,chordata_turdidae/29658536_45510188_eol-full-s...
4,94004bff-3a33-4758-8125-bf72e6e57eab,21252576,7250886,animalia,chordata,indriidae,chordata_indriidae/21252576_7250886_eol-full-s...
...,...,...,...,...,...,...,...
11978,1fa96ea5-32fa-4a25-b8d2-fa99f6e2cb89,29734618,1011315,animalia,chordata,leporidae,chordata_leporidae/29734618_1011315_eol-full-s...
11979,628bf2b4-6ecc-4017-a8e6-4306849e0cfc,29972861,1056842,animalia,chordata,emydidae,chordata_emydidae/29972861_1056842_eol-full-si...
11980,0ecfdec9-b1cd-4d43-96fc-2f8889ec1ad9,30134195,52572074,animalia,chordata,dasyatidae,chordata_dasyatidae/30134195_52572074_eol-full...
11981,27fdb1e9-c5fb-459a-8b6a-6fb222b1c512,9474963,46559139,animalia,chordata,mustelidae,chordata_mustelidae/9474963_46559139_eol-full-...


In the previous notebook we got the data splited.
In this we import it and for the train we import the 3 different strategies we created in the previous notebook. The three of them were tested with the models and the best is kept uncommented.
- original train with no modifications
- train with removal of 'outliers' based on the clip model (best results from the three different approaches tried in the previous notebook)
- train with removal of outliers and augmentation on the minority classes.

In [ ]:
def get_image_paths_and_labels(base_path):
    base = Path(base_path)
    image_paths = list(base.rglob("*.jpg"))  
    data = {
        "filepath": [str(p) for p in image_paths],
        "label": [p.parent.name for p in image_paths]
    }
    return pd.DataFrame(data)

dest_root = Path('.') / "data_split"
#train_df = get_image_paths_and_labels(dest_root / 'train')
train_df = get_image_paths_and_labels(Path('.')/'data_clean_clip')
train_df_aug = get_image_paths_and_labels(Path('.') / 'data_aug_clip')
val_df = get_image_paths_and_labels(dest_root / 'val')
test_df = get_image_paths_and_labels(dest_root / 'test')

<a class="anchor" id="four-bullet"> 
<d style="color:white;">

### 1.3. PreProcessing
</a> 
</d>

After the load of the data we proceed to do the remaining preprocessing steps here. 

#### 1.3.1. Encoder
Encoder: We load a pickle that saves the encoder created in the previous notebook for the train to avoid data leakage. We apply this to the validation and the test dataset so they all have the same values for the same classes.
After this we save the paths and the labels for each of the subset of the data to later use.

In [180]:
encoder = pickle.load(open("encoder.pkl", 'rb'))
val_df["label_encoded"] = encoder.transform(val_df["label"])
test_df["label_encoded"] = encoder.transform(test_df["label"])

In [ ]:
train_labels = tf.constant(train_df["label"].astype(int).values)
train_paths = tf.constant(train_df["filepath"].values)

train_labels_aug = tf.constant(train_df_aug["label"].astype(int).values)
train_paths_aug = tf.constant(train_df_aug["filepath"].astype(str).values)

val_paths = tf.constant(val_df["filepath"].values)
val_labels = tf.constant(val_df["label_encoded"].values)

test_paths = tf.constant(test_df["filepath"].values)
test_labels = tf.constant(test_df["label_encoded"].values)


train_dataset = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_dataset_aug = tf.data.Dataset.from_tensor_slices((train_paths_aug, train_labels_aug))
val_dataset = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
test_dataset = tf.data.Dataset.from_tensor_slices((test_paths, test_labels))

#### 1.3.2. Padding function     
This adds extra pixels to the pictures that are not a specific sizes (0s or a constant value) to avoid distortion of images and loss of information.

In [182]:
def resize_with_padding(image, target_height=224, target_width=224):
    shape = tf.shape(image)[:2] 
    height, width = tf.cast(shape[0], tf.float32), tf.cast(shape[1], tf.float32)

    scale = tf.minimum(target_width / width, target_height / height)
    new_height = tf.cast(height * scale, tf.int32)
    new_width = tf.cast(width * scale, tf.int32)

    resized_image = tf.image.resize(image, [new_height, new_width])

    pad_height = (target_height - new_height) // 2
    pad_width = (target_width - new_width) // 2
    padded_image = tf.image.pad_to_bounding_box(resized_image, pad_height, pad_width, target_height, target_width)

    return padded_image

#### 1.3.3. Preprocessing application
In here we create a function `decode_img` that will apply several different preprocessing steps to the images:
1. Reads the images
2. Decodes them from the JPEG format to be used as a tensor
3. Zooms (crops) the image 80% so that if the animal is too far away it can be read more clearly.
4. Usage of the padding function created previously.
5. Normalization to help the model train better (consistent scale, no bias)
6. Labels turned into one-hot encoding because it works best in NN with multiclass classification.


In [183]:
# Normalizing with 2 different methods to see which one works better
normalization_layer = tf.keras.layers.Rescaling(1./255) # 0-1 normalization
# normalization_layer = tf.keras.layers.Rescaling(1./127.5, offset=-1) # -1 to 1 normalization

def decode_img(img_path, label):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.central_crop(img, central_fraction=0.8)
    img = resize_with_padding(img) 
    #img = tf.image.resize(img, [224, 224]) # this was previously applied but the results were not as good
    img = normalization_layer(img)
    label = tf.one_hot(label, depth=202)
    return img, label


train_dataset = train_dataset.map(decode_img).batch(32).prefetch(tf.data.AUTOTUNE)
train_dataset_aug = train_dataset_aug.map(decode_img).batch(32).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.map(decode_img).batch(32).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.map(decode_img).batch(32).prefetch(tf.data.AUTOTUNE)

<a class="anchor" id="five-bullet"> 
<d style="color:white;">

### 1.4. Augmentation
</a> 
</d>   

Augmentation is important because it gives the model more variety in the images it sees. In the previous notebook it was used to create a dataset with the images augmented proportionally to their class size, and now it will be implemented to the model.

In [ ]:
augmentation_layer = Pipeline([
    RandomBrightness(factor=0.1, value_range=(0.0, 1.0)),
    RandomSaturation(factor=0.2),
    RandomContrast(factor=0.2),
    RandomFlip(),
    RandomRotation(0.1)])

In [ ]:
def show_random_original_and_augmented(dataset, augmentation_layer):
    dataset = dataset.shuffle(1000)  

    for images, _ in dataset.take(1):
        image = images[0]  
        augmented_image = augmentation_layer(tf.expand_dims(image, 0))[0]

        plt.figure(figsize=(10, 5))

        plt.subplot(1, 2, 1)
        plt.imshow(image.numpy())
        plt.title("Original Image")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(augmented_image.numpy())
        plt.title("Augmented Image")
        plt.axis("off")

        plt.show()
        break

In [186]:
#show_random_original_and_augmented(train_dataset, augmentation_layer)

<a class="anchor" id="six-bullet"> 
<d style="color:white;">

### 1.5. Class weights
</a> 
</d>   

Another technique to fight class imbalance tried is the class weights.
This, when applied to the model, makes the images from the minority classes have more weight in the decision.

In [ ]:
train_labels_np = np.array(train_labels)

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels_np),
    y=train_labels_np)

class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}
class_weight_dict

{0: 0.3878825382538254,
 1: 1.8963146314631463,
 2: 0.9752475247524752,
 3: 1.8963146314631463,
 4: 1.8963146314631463,
 5: 0.22021718300862345,
 6: 0.9481573157315731,
 7: 1.8963146314631463,
 8: 1.8963146314631463,
 9: 1.8963146314631463,
 10: 1.8963146314631463,
 11: 1.8963146314631463,
 12: 1.8963146314631463,
 13: 1.8963146314631463,
 14: 1.8963146314631463,
 15: 1.8963146314631463,
 16: 1.8963146314631463,
 17: 0.4876237623762376,
 18: 0.7111179867986799,
 19: 0.9481573157315731,
 20: 2.0078625509609784,
 21: 2.1333539603960396,
 22: 1.8963146314631463,
 23: 0.9481573157315731,
 24: 0.9481573157315731,
 25: 0.3923409582337544,
 26: 0.9481573157315731,
 27: 0.9481573157315731,
 28: 0.9481573157315731,
 29: 2.1333539603960396,
 30: 0.4015725101921957,
 31: 1.8963146314631463,
 32: 1.8963146314631463,
 33: 1.8963146314631463,
 34: 0.9752475247524752,
 35: 1.8963146314631463,
 36: 0.2687690028845404,
 37: 1.8963146314631463,
 38: 0.2994180997047073,
 39: 0.37926292629262925,
 40: 0.9

<hr>
<a class="anchor" id="seven-bullet"> 
<d style="color:white;">

## 2. Models
</a> 
</d>   

<a class="anchor" id="eight-bullet"> 
<d style="color:white;">

### 2.1. PreModel
</a> 
</d>   

Definition of metrics and input shape. Reasons for choice of metrics:

|**Metric used** | **Reason**|
|----------------|----------------------------------------------------------------------------------------------|
|**Accuracy**| Gives an overall idea of performance, but not very good because it's misleading with such an imbalanced dataset.|
|**Precision**| Says how often a model is correct when it says it's a specific class.|
|**Recall**| Tell how many times an image classification is right in comparison to the total there is of that image |
|**F1 Score**| Balances precision and recall. good for our imbalanced 202-class problem.|
|**Loss** (later shown)| Good to see how well the model is learning. The lower the best (model more confident and performative)|

In [ ]:
input_shape = (224, 224, 3)
n_classes = 202

categorical_accuracy = CategoricalAccuracy(name="accuracy")
precision = CategoricalAccuracy(name="precision")
recall = CategoricalAccuracy(name="recall")
f1_score = F1Score(average="macro", name="f1_score")
metrics = [categorical_accuracy,precision, recall, f1_score]

<a class="anchor" id="nine-bullet"> 
<d style="color:white;">

### 2.2. Sequential
</a> 
</d>   

#### **Baseline model**

In [ ]:
model = Sequential(
    layers=[
        Input(shape=input_shape),
        # augmentation_layer, 
        Conv2D(filters=24, kernel_size=(3, 3), activation="relu"),  
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(filters=48, kernel_size=(3, 3), activation="relu"),
        MaxPooling2D(pool_size=(2, 2)),
        Flatten(),
        Dense(n_classes, activation="softmax"),],
    name="sequential")

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=metrics)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_f1_score', 
    mode='max',
    patience=5,
    restore_best_weights=True,
    verbose=1)

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weight_dict)

Epoch 1/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 76s 349ms/step - accuracy: 0.0383 - f1_score: 0.0025 - loss: 6.9085 - precision: 0.0383 - recall: 0.0383 - val_accuracy: 0.0021 - val_f1_score: 2.0932e-05 - val_loss: 5.3113 - val_precision: 0.0021 - val_recall: 0.0021
Epoch 2/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 45s 206ms/step - accuracy: 0.0000e+00 - f1_score: 0.0000e+00 - loss: 5.6113 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.0029 - val_f1_score: 0.0021 - val_loss: 5.3094 - val_precision: 0.0029 - val_recall: 0.0029
Epoch 3/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 46s 212ms/step - accuracy: 0.0544 - f1_score: 0.0090 - loss: 5.4644 - precision: 0.0544 - recall: 0.0544 - val_accuracy: 0.0054 - val_f1_score: 0.0056 - val_loss: 5.3029 - val_precision: 0.0054 - val_recall: 0.0054
Epoch 4/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 46s 211ms/step - accuracy: 0.1002 - f1_score: 0.0346 - loss: 5.3517 - precision: 0.1002 - recall: 0.1002 - val_accuracy: 0.0071 - val_f1_score: 0.0091 - val_loss: 5.3168 - val

In [ ]:
results = model.evaluate(
    test_dataset,
    return_dict=True,
    verbose=0 )

print("Test set results:")
for metric_name, value in results.items():
    print(f"{metric_name}: {value:.4f}")

Test set results:
accuracy: 0.0317
f1_score: 0.0207
loss: 8.8516
precision: 0.0317
recall: 0.0317


#### **Bayesian search**

In [ ]:
# # variable to track best F1 across all trials
# best_f1 = 0.0

# sweep_config = {
#     'method': 'bayes',  
#     'name': 'SequentialModelSweep',
#     'metric': {
#         'name': 'val_f1_score',
#         'goal': 'maximize'
#     },
#     'parameters': {
#         'filters_1': {'values': [16, 32, 48, 64]},
#         'filters_2': {'values': [32, 64, 96, 128]},
#         'kernel_size': {'values': ['3x3', '5x5']},
#         'learning_rate': {'min': 1e-4, 'max': 1e-2},
#         'use_augmentation': {'values': [True, False]},
#         'activation': {'values': ['relu', 'leaky_relu']},
#         'optimizer': {'values': ['adam', 'rmsprop']},
#     }
# }

# # Initialize sweep
# sweep_id = wandb.sweep(sweep_config, project="dl_project")

# def train_model(config=None):
#     with wandb.init(config=config):
#         config = wandb.config

#         model = Sequential()
#         model.add(Input(shape=input_shape))

#         if config.use_augmentation:
#             model.add(augmentation_layer)

#         kernel_size = tuple(map(int, config.kernel_size.split("x")))
#         model.add(Conv2D(config.filters_1, kernel_size=kernel_size))
#         model.add(LeakyReLU() if config.activation == "leaky_relu" else tf.keras.layers.Activation(config.activation))
#         model.add(MaxPooling2D(pool_size=(2, 2)))

#         model.add(Conv2D(config.filters_2, kernel_size=kernel_size))
#         model.add(LeakyReLU() if config.activation == "leaky_relu" else tf.keras.layers.Activation(config.activation))
#         model.add(MaxPooling2D(pool_size=(2, 2)))

#         model.add(Flatten())
#         model.add(Dense(n_classes, activation="softmax"))

#         optimizer = Adam(learning_rate=config.learning_rate) if config.optimizer == "adam" else RMSprop(learning_rate=config.learning_rate)

#         model.compile(optimizer=optimizer, loss=CategoricalCrossentropy(), metrics=metrics)

#         early_stopping = EarlyStopping(monitor="val_f1_score", patience=3, restore_best_weights=True, mode="max")
#         wandb_callback = WandbCallback(save_graph=False, save_model=False)

#         history = model.fit(
#             train_dataset,
#             validation_data=val_dataset,
#             epochs=5,
#             class_weight=class_weight_dict,
#             callbacks=[early_stopping, wandb_callback]
#         )

#         global best_f1
#         val_f1_scores = history.history.get("val_f1_score")
#         if val_f1_scores:
#             trial_best = max(val_f1_scores)
#             if trial_best > best_f1:
#                 best_f1 = trial_best
#                 model.save("best_model.keras")
#                 print(f"\n Saved new best model with val_f1_score = {best_f1:.4f}")

# wandb.agent(sweep_id, function=train_model, count=50)

In [ ]:
# # check best model parameters
# api = wandb.Api()
# sweep = api.sweep("matildestreet-nova-ims/dl_project/4xk38eju")
# runs = sweep.runs
# best_run = max(runs, key=lambda r: r.summary.get("val_f1_score", 0))

# print("Best run config:")
# for key, value in best_run.config.items():
#     print(f"{key}: {value}")

# # output - best was trial 49
# Best run config:
# filters_1: 32
# filters_2: 32
# optimizer: rmsprop
# activation: leaky_relu
# kernel_size: 5x5
# learning_rate: 0.001526590318714996
# use_augmentation: False

In [ ]:
best_model_seq = Sequential(
    layers=[
        Input(shape=input_shape),
        Conv2D(filters=32, kernel_size=(5, 5), activation="leaky_relu"),  
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(filters=32, kernel_size=(5, 5), activation="leaky_relu"),
        MaxPooling2D(pool_size=(2, 2)),
        Flatten(),
        Dense(n_classes, activation="softmax"),
    ],
    name="sequential"
)

In [ ]:
optimizer = RMSprop(learning_rate=0.001526590318714996)

best_model_seq.compile(
    optimizer=optimizer,
    loss=CategoricalCrossentropy(),
    metrics=metrics)

In [ ]:
history = best_model_seq.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs= 10,
    class_weight=class_weight_dict)

Epoch 1/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 59s 269ms/step - accuracy: 0.1193 - f1_score: 0.0477 - loss: 8.7189 - precision: 0.1193 - recall: 0.1193 - val_accuracy: 0.0125 - val_f1_score: 0.0051 - val_loss: 5.4196 - val_precision: 0.0125 - val_recall: 0.0125
Epoch 2/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 162s 751ms/step - accuracy: 0.2406 - f1_score: 0.0659 - loss: 5.4215 - precision: 0.2406 - recall: 0.2406 - val_accuracy: 0.0025 - val_f1_score: 2.4752e-05 - val_loss: 27.5847 - val_precision: 0.0025 - val_recall: 0.0025
Epoch 3/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 53s 242ms/step - accuracy: 0.3017 - f1_score: 0.1059 - loss: 5.1252 - precision: 0.3017 - recall: 0.3017 - val_accuracy: 0.0042 - val_f1_score: 0.0047 - val_loss: 19.4078 - val_precision: 0.0042 - val_recall: 0.0042
Epoch 4/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 57s 262ms/step - accuracy: 0.3925 - f1_score: 0.1587 - loss: 4.3590 - precision: 0.3925 - recall: 0.3925 - val_accuracy: 0.0226 - val_f1_score: 0.0182 - val_loss: 7.5105 - val_precision: 0

In [ ]:
results = best_model_seq.evaluate(
    test_dataset,
    return_dict=True,
    verbose=0)

print("Test set results:")
for metric_name, value in results.items():
    print(f"{metric_name}: {value:.4f}")

Test set results:
accuracy: 0.0317
f1_score: 0.0319
loss: 22.7174
precision: 0.0317
recall: 0.0317


the f1 score is higher after the bayesian optimization, however the difference is not that big and the loss is much higher

#### **evaluate impact of using class_weight_dict**

In [ ]:
best_model_seq_noweights = Sequential(
    layers=[
        Input(shape=input_shape),
        Conv2D(filters=32, kernel_size=(5, 5), activation="leaky_relu"),  
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(filters=32, kernel_size=(5, 5), activation="leaky_relu"),
        MaxPooling2D(pool_size=(2, 2)),
        Flatten(),
        Dense(n_classes, activation="softmax"),
    ],
    name="sequential"
)

In [ ]:
optimizer = RMSprop(learning_rate=0.001526590318714996)

best_model_seq_noweights.compile(
    optimizer=optimizer,
    loss=CategoricalCrossentropy(),
    metrics=metrics)

In [ ]:
history = best_model_seq_noweights.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs= 10)

Epoch 1/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 86s 396ms/step - accuracy: 0.1320 - f1_score: 0.0556 - loss: 8.7139 - precision: 0.1320 - recall: 0.1320 - val_accuracy: 0.0042 - val_f1_score: 0.0027 - val_loss: 6.2600 - val_precision: 0.0042 - val_recall: 0.0042
Epoch 2/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 56s 260ms/step - accuracy: 0.2833 - f1_score: 0.0758 - loss: 4.7317 - precision: 0.2833 - recall: 0.2833 - val_accuracy: 0.0054 - val_f1_score: 0.0052 - val_loss: 7.5460 - val_precision: 0.0054 - val_recall: 0.0054
Epoch 3/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 61s 283ms/step - accuracy: 0.3087 - f1_score: 0.1038 - loss: 4.4561 - precision: 0.3087 - recall: 0.3087 - val_accuracy: 0.0042 - val_f1_score: 0.0040 - val_loss: 11.9874 - val_precision: 0.0042 - val_recall: 0.0042
Epoch 4/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 55s 255ms/step - accuracy: 0.4009 - f1_score: 0.1563 - loss: 3.9770 - precision: 0.4009 - recall: 0.4009 - val_accuracy: 0.0029 - val_f1_score: 0.0014 - val_loss: 31.0096 - val_precision: 0.0029

In [ ]:
results = best_model_seq_noweights.evaluate(
    test_dataset,
    return_dict=True,
    verbose=0)

print("Test set results:")
for metric_name, value in results.items():
    print(f"{metric_name}: {value:.4f}")

Test set results:
accuracy: 0.0246
f1_score: 0.0215
loss: 21.2632
precision: 0.0246
recall: 0.0246


It performs better with class weights

#### **Trying with the dataset with augmentation to see if the score gets better**

In [ ]:
best_model_seq_aug = Sequential(
    layers=[
        Input(shape=input_shape),
        Conv2D(filters=32, kernel_size=(5, 5), activation="leaky_relu"),  
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(filters=32, kernel_size=(5, 5), activation="leaky_relu"),
        MaxPooling2D(pool_size=(2, 2)),
        Flatten(),
        Dense(n_classes, activation="softmax"),
    ],
    name="sequential")

In [ ]:
optimizer = RMSprop(learning_rate=0.001526590318714996)

best_model_seq_aug.compile(
    optimizer=optimizer,
    loss=CategoricalCrossentropy(),
    metrics=metrics)

In [ ]:
history = best_model_seq_aug.fit(
    train_dataset_aug,
    validation_data=val_dataset,
    epochs= 10,
    class_weight=class_weight_dict)

Epoch 1/10
1137/1137 ━━━━━━━━━━━━━━━━━━━━ 246s 216ms/step - accuracy: 0.5834 - f1_score: 0.3526 - loss: 5.4998 - precision: 0.5834 - recall: 0.5834 - val_accuracy: 0.0025 - val_f1_score: 2.4773e-05 - val_loss: 12.4778 - val_precision: 0.0025 - val_recall: 0.0025
Epoch 2/10
1137/1137 ━━━━━━━━━━━━━━━━━━━━ 247s 217ms/step - accuracy: 0.7352 - f1_score: 0.3719 - loss: 4.6937 - precision: 0.7352 - recall: 0.7352 - val_accuracy: 0.0025 - val_f1_score: 2.4752e-05 - val_loss: 22.5461 - val_precision: 0.0025 - val_recall: 0.0025
Epoch 3/10
1137/1137 ━━━━━━━━━━━━━━━━━━━━ 243s 213ms/step - accuracy: 0.7431 - f1_score: 0.3778 - loss: 4.5749 - precision: 0.7431 - recall: 0.7431 - val_accuracy: 0.0025 - val_f1_score: 2.4773e-05 - val_loss: 18.5200 - val_precision: 0.0025 - val_recall: 0.0025
Epoch 4/10
1137/1137 ━━━━━━━━━━━━━━━━━━━━ 244s 214ms/step - accuracy: 0.7477 - f1_score: 0.3774 - loss: 4.4094 - precision: 0.7477 - recall: 0.7477 - val_accuracy: 0.0025 - val_f1_score: 2.4763e-05 - val_loss: 2

In [ ]:
results = best_model_seq_aug.evaluate(
    test_dataset,
    return_dict=True,
    verbose=0 )

print("Test set results:")
for metric_name, value in results.items():
    print(f"{metric_name}: {value:.4f}")

Test set results:
accuracy: 0.0062
f1_score: 0.0065
loss: 19.5562
precision: 0.0062
recall: 0.0062


Applying data augmentation to the minority classes decreased significantly the f1 score

The best sequential model we obtained is after the bayesian search with the class weights we defined and without the augmented dataset. Let´s train it with 50 epochs to see how much it will improve.

In [ ]:
best_model_seq = Sequential(
    layers=[
        Input(shape=input_shape),
        Conv2D(filters=32, kernel_size=(5, 5), activation="leaky_relu"),  
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(filters=32, kernel_size=(5, 5), activation="leaky_relu"),
        MaxPooling2D(pool_size=(2, 2)),
        Flatten(),
        Dense(n_classes, activation="softmax"),
    ],
    name="sequential")

optimizer = RMSprop(learning_rate=0.001526590318714996)

best_model_seq.compile(
    optimizer=optimizer,
    loss=CategoricalCrossentropy(),
    metrics=metrics)

history = best_model_seq.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs= 50,
    class_weight=class_weight_dict)

Epoch 1/50
216/216 ━━━━━━━━━━━━━━━━━━━━ 55s 250ms/step - accuracy: 0.1039 - f1_score: 0.0415 - loss: 9.4057 - precision: 0.1039 - recall: 0.1039 - val_accuracy: 0.0092 - val_f1_score: 0.0052 - val_loss: 5.3550 - val_precision: 0.0092 - val_recall: 0.0092
Epoch 2/50
216/216 ━━━━━━━━━━━━━━━━━━━━ 58s 268ms/step - accuracy: 0.2614 - f1_score: 0.0721 - loss: 5.6423 - precision: 0.2614 - recall: 0.2614 - val_accuracy: 0.0058 - val_f1_score: 0.0030 - val_loss: 6.4028 - val_precision: 0.0058 - val_recall: 0.0058
Epoch 3/50
216/216 ━━━━━━━━━━━━━━━━━━━━ 59s 273ms/step - accuracy: 0.3028 - f1_score: 0.1006 - loss: 5.0192 - precision: 0.3028 - recall: 0.3028 - val_accuracy: 0.0217 - val_f1_score: 0.0156 - val_loss: 8.3172 - val_precision: 0.0217 - val_recall: 0.0217
Epoch 4/50
216/216 ━━━━━━━━━━━━━━━━━━━━ 56s 257ms/step - accuracy: 0.3853 - f1_score: 0.1480 - loss: 4.3567 - precision: 0.3853 - recall: 0.3853 - val_accuracy: 0.0305 - val_f1_score: 0.0164 - val_loss: 7.8310 - val_precision: 0.0305 -

In [ ]:
results = best_model_seq.evaluate(
    test_dataset,
    return_dict=True,
    verbose=0 )

print("Test set results:")
for metric_name, value in results.items():
    print(f"{metric_name}: {value:.4f}")

Test set results:
accuracy: 0.0421
f1_score: 0.0364
loss: 179.1308
precision: 0.0421
recall: 0.0421


<a class="anchor" id="ten-bullet"> 
<d style="color:white;">

### 2.3. Functional
</a> 
</d>   

**Baseline model**

In [211]:
def functional_model():
    conv_layer_1 = layers.Conv2D(
        filters=3 * 8,
        kernel_size=(3, 3),
        activation="relu",
        name="conv_layer_1"
    )
    max_pool_layer_1 = layers.MaxPooling2D(pool_size=(2, 2), name="max_pool_layer_1")

    conv_layer_2 = layers.Conv2D(
        filters=3 * 16,
        kernel_size=(3, 3),
        name="conv_layer_2"
    )
    act_layer_2 = layers.LeakyReLU(negative_slope=0.3, name="act_layer_2")
    max_pool_layer_2 = layers.MaxPooling2D(pool_size=(2, 2), name="max_pool_layer_2")

    flatten_layer = layers.Flatten(name="flatten_layer")
    dense_layer = layers.Dense(
        n_classes,
        activation="softmax",
        name="classification_head"
    )

    inputs = layers.Input(shape=input_shape)
    x = inputs

    x = conv_layer_1(x)
    x = max_pool_layer_1(x)

    x = conv_layer_2(x)
    x = act_layer_2(x)
    x = max_pool_layer_2(x)

    x = flatten_layer(x)
    x = dense_layer(x)

    outputs = x

    return models.Model(inputs=inputs, outputs=outputs, name="my_tiny_functional_cnn")

In [ ]:
model=functional_model()
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=metrics)

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weight_dict)

Epoch 1/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 47s 214ms/step - accuracy: 0.0422 - f1_score: 0.0250 - loss: 8.1568 - precision: 0.0422 - recall: 0.0422 - val_accuracy: 0.0113 - val_f1_score: 0.0014 - val_loss: 6.6526 - val_precision: 0.0113 - val_recall: 0.0113
Epoch 2/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 47s 217ms/step - accuracy: 0.0016 - f1_score: 3.4470e-04 - loss: 6.0699 - precision: 0.0016 - recall: 0.0016 - val_accuracy: 0.0138 - val_f1_score: 0.0054 - val_loss: 7.8069 - val_precision: 0.0138 - val_recall: 0.0138
Epoch 3/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 46s 212ms/step - accuracy: 0.0143 - f1_score: 0.0098 - loss: 5.5436 - precision: 0.0143 - recall: 0.0143 - val_accuracy: 0.0121 - val_f1_score: 0.0016 - val_loss: 9.6661 - val_precision: 0.0121 - val_recall: 0.0121
Epoch 4/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 46s 213ms/step - accuracy: 0.0561 - f1_score: 0.0356 - loss: 4.9207 - precision: 0.0561 - recall: 0.0561 - val_accuracy: 0.0121 - val_f1_score: 0.0108 - val_loss: 8.6470 - val_precision: 0.01

In [ ]:
results = model.evaluate(
    test_dataset,
    return_dict=True,
    verbose=0)

print("Test set results:")
for metric_name, value in results.items():
    print(f"{metric_name}: {value:.4f}")

Test set results:
accuracy: 0.0346
f1_score: 0.0287
loss: 13.0643
precision: 0.0346
recall: 0.0346


**Bayesian search**

In [ ]:
# sweep_config = {
#     'method': 'bayes',
#     'name': 'FunctionalModelSweep',
#     'metric': {
#         'name': 'val_f1_score',
#         'goal': 'maximize'
#     },
#     'parameters': {
#         'filters_1': {'values': [16, 32, 48, 64]},
#         'filters_2': {'values': [32, 64, 96, 128]},
#         'kernel_size': {'values': ['3x3', '5x5']},
#         'learning_rate': {'min': 1e-4, 'max': 1e-2},
#         'use_augmentation': {'values': [True, False]},
#         'activation': {'values': ['relu', 'leaky_relu']},
#         'alpha': {'min': 0.1, 'max': 0.5},  # for LeakyReLU
#         'optimizer': {'values': ['adam', 'rmsprop']},
#     }
# }

# sweep_id = wandb.sweep(sweep_config, project="dl_project")

# def train_model(config=None):
#     with wandb.init(config=config):
#         config = wandb.config

#         conv1 = layers.Conv2D(filters=config.filters_1, kernel_size=tuple(map(int, config.kernel_size.split('x'))), name='conv_1')
#         conv2 = layers.Conv2D(filters=config.filters_2, kernel_size=tuple(map(int, config.kernel_size.split('x'))), name='conv_2')
#         maxpool = layers.MaxPooling2D(pool_size=(2, 2))

#         activation_1 = layers.Activation('relu') if config.activation == "relu" else layers.LeakyReLU(alpha=config.alpha)
#         activation_2 = layers.Activation('relu') if config.activation == "relu" else layers.LeakyReLU(alpha=config.alpha)

#         flatten = layers.Flatten()
#         dense = layers.Dense(n_classes, activation='softmax', name='output')

#         inputs = layers.Input(shape=input_shape)
#         x = inputs

#         if config.use_augmentation:
#             x = augmentation_layer(x)

#         x = conv1(x)
#         x = maxpool(x)

#         x = conv2(x)
#         x = activation_2(x)
#         x = maxpool(x)

#         x = flatten(x)
#         outputs = dense(x)

#         model = models.Model(inputs=inputs, outputs=outputs)

#         optimizer = Adam(learning_rate=config.learning_rate) if config.optimizer == "adam" else RMSprop(learning_rate=config.learning_rate)
#         model.compile(optimizer=optimizer, loss=CategoricalCrossentropy(), metrics=metrics)

#         early_stopping = EarlyStopping(monitor='val_f1_score', patience=3, restore_best_weights=True, mode='max')

#         model.fit(
#             train_dataset,
#             validation_data=val_dataset,
#             epochs=10,
#             class_weight=class_weight_dict,
#             callbacks=[early_stopping, WandbCallback(save_model=False, save_graph=False)],
#             verbose=1
#         )

# wandb.agent(sweep_id, function=train_model, count=50)

In [ ]:
# api = wandb.Api()
# sweep = api.sweep("matildestreet-nova-ims/dl_project/7jkmks1l")  
# runs = sweep.runs

# best_run = max(runs, key=lambda r: r.summary.get("val_f1_score", 0))

# print("Best run config:")
# for key, value in best_run.config.items():
#     print(f"{key}: {value}")

# Best run config:
# alpha: 0.48096970546631235
# filters_1: 64
# filters_2: 32
# optimizer: rmsprop
# activation: leaky_relu
# kernel_size: 3x3
# learning_rate: 0.0005838087914898498
# use_augmentation: False

In [ ]:
def best_functional_model():
    inputs = layers.Input(shape=input_shape, name="input")

    x = layers.Conv2D(64, kernel_size=(3, 3), activation=None, name="conv_1")(inputs)
    x = layers.LeakyReLU(alpha=0.48, name="leaky_relu_1")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), name="max_pool_1")(x)

    x = layers.Conv2D(32, kernel_size=(3, 3), activation=None, name="conv_2")(x)
    x = layers.LeakyReLU(alpha=0.48, name="leaky_relu_2")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), name="max_pool_2")(x)

    x = layers.Flatten(name="flatten")(x)
    outputs = layers.Dense(n_classes, activation="softmax", name="output")(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="best_functional_model")
    return model

best_model = best_functional_model()
optimizer = optimizers.RMSprop(learning_rate=0.0005838087914898498)

best_model.compile(
    optimizer=optimizer,
    loss=CategoricalCrossentropy(),
    metrics=metrics)

history = best_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weight_dict)

Epoch 1/10


c:\USERS\PAULO\ANACONDA3\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


216/216 ━━━━━━━━━━━━━━━━━━━━ 70s 322ms/step - accuracy: 0.1183 - f1_score: 0.0501 - loss: 5.5222 - precision: 0.1183 - recall: 0.1183 - val_accuracy: 0.0430 - val_f1_score: 0.0048 - val_loss: 5.2530 - val_precision: 0.0430 - val_recall: 0.0430
Epoch 2/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 80s 313ms/step - accuracy: 0.2458 - f1_score: 0.0645 - loss: 4.5169 - precision: 0.2458 - recall: 0.2458 - val_accuracy: 0.0054 - val_f1_score: 0.0045 - val_loss: 5.6101 - val_precision: 0.0054 - val_recall: 0.0054
Epoch 3/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 69s 319ms/step - accuracy: 0.3357 - f1_score: 0.1046 - loss: 3.7915 - precision: 0.3357 - recall: 0.3357 - val_accuracy: 0.0050 - val_f1_score: 0.0050 - val_loss: 6.4653 - val_precision: 0.0050 - val_recall: 0.0050
Epoch 4/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 69s 318ms/step - accuracy: 0.3999 - f1_score: 0.1471 - loss: 3.1366 - precision: 0.3999 - recall: 0.3999 - val_accuracy: 0.0188 - val_f1_score: 0.0139 - val_loss: 6.7894 - val_precision: 0.0188 - val_recall

In [ ]:
results = best_model.evaluate(test_dataset, return_dict=True, verbose=0)
print("\nTest set results:")
for metric_name, value in results.items():
    print(f"{metric_name}: {value:.4f}")


Test set results:
accuracy: 0.0354
f1_score: 0.0350
loss: 7.7662
precision: 0.0354
recall: 0.0354


Let´s train it with 50 epochs to see how much it will improve.

In [ ]:
def best_functional_model():
    inputs = layers.Input(shape=input_shape, name="input")

    x = layers.Conv2D(64, kernel_size=(3, 3), activation=None, name="conv_1")(inputs)
    x = layers.LeakyReLU(alpha=0.48, name="leaky_relu_1")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), name="max_pool_1")(x)

    x = layers.Conv2D(32, kernel_size=(3, 3), activation=None, name="conv_2")(x)
    x = layers.LeakyReLU(alpha=0.48, name="leaky_relu_2")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), name="max_pool_2")(x)

    x = layers.Flatten(name="flatten")(x)
    outputs = layers.Dense(n_classes, activation="softmax", name="output")(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="best_functional_model")
    return model

best_model = best_functional_model()
optimizer = optimizers.RMSprop(learning_rate=0.0005838087914898498)

best_model.compile(
    optimizer=optimizer,
    loss=CategoricalCrossentropy(),
    metrics=metrics)

history = best_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,
    class_weight=class_weight_dict)

Epoch 1/50
216/216 ━━━━━━━━━━━━━━━━━━━━ 67s 305ms/step - accuracy: 0.1210 - f1_score: 0.0556 - loss: 5.5910 - precision: 0.1210 - recall: 0.1210 - val_accuracy: 0.0175 - val_f1_score: 0.0052 - val_loss: 5.2407 - val_precision: 0.0175 - val_recall: 0.0175
Epoch 2/50
216/216 ━━━━━━━━━━━━━━━━━━━━ 68s 313ms/step - accuracy: 0.2567 - f1_score: 0.0699 - loss: 4.5655 - precision: 0.2567 - recall: 0.2567 - val_accuracy: 0.0050 - val_f1_score: 0.0015 - val_loss: 5.6671 - val_precision: 0.0050 - val_recall: 0.0050
Epoch 3/50
216/216 ━━━━━━━━━━━━━━━━━━━━ 68s 316ms/step - accuracy: 0.3478 - f1_score: 0.1102 - loss: 3.7910 - precision: 0.3478 - recall: 0.3478 - val_accuracy: 0.0171 - val_f1_score: 0.0085 - val_loss: 6.7680 - val_precision: 0.0171 - val_recall: 0.0171
Epoch 4/50
216/216 ━━━━━━━━━━━━━━━━━━━━ 68s 315ms/step - accuracy: 0.4191 - f1_score: 0.1510 - loss: 3.1192 - precision: 0.4191 - recall: 0.4191 - val_accuracy: 0.0201 - val_f1_score: 0.0145 - val_loss: 7.2763 - val_precision: 0.0201 -

In [ ]:
results = best_model.evaluate(test_dataset, return_dict=True, verbose=0)
print("\nTest set results:")
for metric_name, value in results.items():
    print(f"{metric_name}: {value:.4f}")


Test set results:
accuracy: 0.0516
f1_score: 0.0438
loss: 20.8757
precision: 0.0516
recall: 0.0516


<a class="anchor" id="eleven-bullet"> 
<d style="color:white;">

### 2.4. Functional with more layers
</a> 
</d>  

In [227]:
def func_model_more_layers(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape, name="input_layer")

    x = layers.Conv2D(128, 3, strides=2, padding="same", name="entry_conv")(inputs)
    x = layers.BatchNormalization(name="entry_bn")(x)
    x = layers.Activation("relu", name="entry_relu")(x)

    previous_block_activation = x

    for i, size in enumerate([256, 512, 728]):
        x = layers.Activation("relu", name=f"block_{i+1}_act_1")(x)
        x = layers.SeparableConv2D(size, 3, padding="same", name=f"block_{i+1}_sepconv_1")(x)
        x = layers.BatchNormalization(name=f"block_{i+1}_bn_1")(x)

        x = layers.Activation("relu", name=f"block_{i+1}_act_2")(x)
        x = layers.SeparableConv2D(size, 3, padding="same", name=f"block_{i+1}_sepconv_2")(x)
        x = layers.BatchNormalization(name=f"block_{i+1}_bn_2")(x)

        x = layers.MaxPooling2D(3, strides=2, padding="same", name=f"block_{i+1}_pool")(x)

        residual = layers.Conv2D(size, 1, strides=2, padding="same", name=f"block_{i+1}_residual")(previous_block_activation)
        x = layers.add([x, residual], name=f"block_{i+1}_add")
        previous_block_activation = x

    x = layers.SeparableConv2D(1024, 3, padding="same", name="final_sepconv")(x)
    x = layers.BatchNormalization(name="final_bn")(x)
    x = layers.Activation("relu", name="final_relu")(x)

    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = layers.Dropout(0.25, name="dropout")(x)

    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    return keras.Model(inputs=inputs, outputs=outputs, name="deeper_residual_model")

model = func_model_more_layers(input_shape=input_shape, num_classes=n_classes)

optimizer = optimizers.RMSprop(learning_rate=0.0005838087914898498)

model.compile(
    optimizer=optimizer,
    loss=CategoricalCrossentropy(),
    metrics=metrics)

keras.utils.plot_model(model, show_shapes=True, to_file="func_model_more_layers.png")

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weight_dict)

results = model.evaluate(test_dataset, return_dict=True, verbose=0)
print("\nTest set results:")
for metric_name, value in results.items():
    print(f"{metric_name}: {value:.4f}")

You must install pydot (`pip install pydot`) for `plot_model` to work.
Epoch 1/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 1719s 8s/step - accuracy: 0.0162 - f1_score: 0.0020 - loss: 6.4733 - precision: 0.0162 - recall: 0.0162 - val_accuracy: 0.0100 - val_f1_score: 2.3203e-04 - val_loss: 5.2978 - val_precision: 0.0100 - val_recall: 0.0100
Epoch 2/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 1675s 8s/step - accuracy: 0.0070 - f1_score: 0.0011 - loss: 6.1792 - precision: 0.0070 - recall: 0.0070 - val_accuracy: 0.0242 - val_f1_score: 2.3420e-04 - val_loss: 5.2806 - val_precision: 0.0242 - val_recall: 0.0242
Epoch 3/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 1667s 8s/step - accuracy: 0.0071 - f1_score: 2.5702e-04 - loss: 6.0084 - precision: 0.0071 - recall: 0.0071 - val_accuracy: 0.0242 - val_f1_score: 2.3779e-04 - val_loss: 5.5273 - val_precision: 0.0242 - val_recall: 0.0242
Epoch 4/10
216/216 ━━━━━━━━━━━━━━━━━━━━ 809s 4s/step - accuracy: 0.0079 - f1_score: 7.9451e-04 - loss: 5.9137 - precision: 0.0079 - recall: 0.0079 - va